# Wine Quality Prediction using Support Vector Machine (SVM)
**Dataset:** Wine Quality Dataset (Kaggle)  


## 1. Import Required Libraries

In [ ]:
# ── Standard Libraries ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn: Preprocessing ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── Scikit-learn: SVM ────────────────────────────────────────────────────────
from sklearn.svm import SVC

# ── Scikit-learn: Evaluation Metrics ─────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

# ── Dimensionality Reduction ──────────────────────────────────────────────────
from sklearn.decomposition import PCA

print("All libraries imported successfully!")

## 2. Dataset Description

**Dataset:** Wine Quality (Combined Red & White)  
**Size:** 6,497 instances, 13 attributes



#### Load Dataset

In [ ]:
df = pd.read_csv('winequalityN.csv')

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

#### Basic Info

In [ ]:
print("Dataset Info:")
df.info()

print("\nStatistical Summary:")
df.describe()

## 3. Exploratory Data Analysis (EDA)

#### Check Missing Values and Quality Distribution

In [ ]:
print("Missing Values per Column:")
print(df.isnull().sum())

# ────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
df['quality'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribution of Wine Quality Scores')
plt.xlabel('Quality Score')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
df['type'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
plt.title('Wine Type Distribution')
plt.ylabel('')

plt.tight_layout()
plt.show()

#### Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
numeric_df = df.drop(columns=['type'])  # drop categorical for correlation
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

#### Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
df[numeric_cols].hist(bins=20, figsize=(15, 10), color='teal', edgecolor='black')
plt.suptitle('Histogram of All Numeric Features', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

#### Boxplot: Alcohol vs Quality

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x='quality', y='alcohol', data=df, palette='Set2')
plt.title('Alcohol Content vs Wine Quality')
plt.xlabel('Quality Score')
plt.ylabel('Alcohol (%)')
plt.show()


## 4. Data Preprocessing

#### Step 1: Handle Missing Values

In [ ]:
# Fill missing numeric values with column median (robust to outliers)
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)

print("Missing values after imputation:", df.isnull().sum().sum())

#### Step 2: Encode Categorical Variable (type: red/white)

In [ ]:
le = LabelEncoder()
df['type_encoded'] = le.fit_transform(df['type'])  # red=1, white=0 (or vice versa)
print("Encoding mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

#### Step 3: Binarize Target – Good (1) vs Bad (0)

In [ ]:
# Quality >= 7 → Good Wine (1) | Quality < 7 → Average/Poor Wine (0)
# This converts a multi-class problem into a cleaner binary classification task
df['quality_label'] = (df['quality'] >= 7).astype(int)

print("Class Distribution:")
print(df['quality_label'].value_counts())
print("\n0 = Average/Poor Wine | 1 = Good Wine")

plt.figure(figsize=(5,4))
df['quality_label'].value_counts().plot(kind='bar', color=['salmon','mediumseagreen'], edgecolor='black')
plt.xticks([0,1], ['Poor/Average (0)', 'Good (1)'], rotation=0)
plt.title('Binary Class Distribution')
plt.ylabel('Count')
plt.show()

#### Step 4: Define Features (X) and Target (y)

In [ ]:
feature_cols = [
    'type_encoded', 'fixed acidity', 'volatile acidity', 'citric acid',
    'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
    'density', 'pH', 'sulphates', 'alcohol'
]

X = df[feature_cols]
y = df['quality_label']

print("Feature matrix shape:", X.shape)
print("Target vector shape: ", y.shape)

#### Step 5: Train-Test Split (80% train, 20% test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

#### Step 6: Feature Scaling (CRITICAL for SVM)

In [ ]:
# SVM is distance-based → features must be on the same scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled  = scaler.transform(X_test)         # apply same scale to test

print("Scaling complete. Sample mean (should be ~0):", X_train_scaled.mean().round(4))
print("Sample std  (should be ~1):", X_train_scaled.std().round(4))

## 5. Dimensionality Reduction (PCA)

#### PCA: Explained Variance Analysis

In [ ]:
pca_full = PCA()
pca_full.fit(X_train_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, marker='o', color='steelblue')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% variance threshold')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA: Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.show()

# Find components needed for 95% variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"Components needed for 95% variance: {n_components_95}")

#### NOTE: We will train SVM on FULL scaled features (better accuracy)

In [ ]:
# PCA was used only for analysis. Full features retained for SVM training.
print("SVM will be trained on all", X_train_scaled.shape[1], "scaled features.")

## 7. Model Training — SVM with Different Kernels

In [ ]:
# ── Train SVM with Multiple Kernels for Comparison ───────────────────────────
kernels = ['linear', 'rbf', 'poly']
results = {}

for kernel in kernels:
    print(f"Training SVM with '{kernel}' kernel...")
    svm_model = SVC(kernel=kernel, random_state=42, probability=True)
    svm_model.fit(X_train_scaled, y_train)

    y_pred  = svm_model.predict(X_test_scaled)
    acc     = accuracy_score(y_test, y_pred)
    cv_score = cross_val_score(svm_model, X_train_scaled, y_train, cv=5).mean()

    results[kernel] = {
        'model': svm_model,
        'predictions': y_pred,
        'accuracy': acc,
        'cv_score': cv_score
    }
    print(f"  → Test Accuracy: {acc:.4f} | CV Score: {cv_score:.4f}\n")

print("All kernels trained!")

## 8. Hyperparameter Tuning (GridSearchCV)

In [ ]:
# ── Grid Search for Best C and Gamma with RBF Kernel ─────────────────────────
# Uses 5-fold cross-validation to find the best parameters
param_grid = {
    'C':     [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf']
}

print("Running Grid Search... (this may take a few minutes)")
grid_search = GridSearchCV(
    SVC(random_state=42, probability=True),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,    # use all CPU cores
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print("\nBest Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", round(grid_search.best_score_, 4))

In [ ]:
# ── Best Model from GridSearch ────────────────────────────────────────────────
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)

best_accuracy = accuracy_score(y_test, y_pred_best)
print(f"Best SVM Test Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")

## 9. Results & Evaluation

In [ ]:
# ── Classification Report ─────────────────────────────────────────────────────
print("=" * 55)
print("       CLASSIFICATION REPORT (Best SVM - RBF)")
print("=" * 55)
print(classification_report(y_test, y_pred_best,
      target_names=['Poor/Average', 'Good Wine']))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Poor/Average', 'Good'])
disp.plot(cmap='Blues', colorbar=False)
plt.title('Confusion Matrix — Best SVM (RBF Kernel)')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (TN) : {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP) : {tp}")

In [ ]:
# ── Kernel Comparison Bar Chart ───────────────────────────────────────────────
kernel_names  = list(results.keys()) + ['RBF (tuned)']
accuracies    = [results[k]['accuracy'] for k in results] + [best_accuracy]
cv_scores     = [results[k]['cv_score'] for k in results] + [grid_search.best_score_]

x = np.arange(len(kernel_names))
width = 0.35

plt.figure(figsize=(10, 5))
bars1 = plt.bar(x - width/2, accuracies, width, label='Test Accuracy',  color='steelblue')
bars2 = plt.bar(x + width/2, cv_scores,  width, label='CV Score (mean)', color='coral')

for bar in bars1 + bars2:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.xticks(x, kernel_names)
plt.ylim(0.75, 1.0)
plt.ylabel('Score')
plt.title('SVM Kernel Comparison: Test Accuracy vs Cross-Validation Score')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance via Linear SVM Coefficients ───────────────────────────
# Train a Linear SVM to extract feature weights (as proxy for importance)
svm_linear = SVC(kernel='linear', C=1, random_state=42)
svm_linear.fit(X_train_scaled, y_train)

feature_weights = pd.Series(
    np.abs(svm_linear.coef_[0]),
    index=feature_cols
).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feature_weights.plot(kind='barh', color='mediumslateblue', edgecolor='black')
plt.title('Feature Importance (Linear SVM Coefficients)')
plt.xlabel('Absolute Coefficient Value')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cross-Validation Scores (Best Model) ─────────────────────────────────────
cv_scores_best = cross_val_score(best_svm, X_train_scaled, y_train, cv=10)

print("10-Fold Cross Validation Scores:")
print(cv_scores_best.round(4))
print(f"\nMean CV Accuracy : {cv_scores_best.mean():.4f}")
print(f"Std Dev          : {cv_scores_best.std():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), cv_scores_best, marker='o', linestyle='-', color='teal')
plt.axhline(cv_scores_best.mean(), color='red', linestyle='--', label=f'Mean = {cv_scores_best.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('10-Fold Cross-Validation Accuracy (Best SVM)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 10. Summary of Results

In [ ]:
# ── Final Results Summary Table ───────────────────────────────────────────────
summary = pd.DataFrame({
    'Kernel': ['Linear', 'RBF (default)', 'Polynomial', 'RBF (tuned)'],
    'Test Accuracy': [
        round(results['linear']['accuracy'], 4),
        round(results['rbf']['accuracy'], 4),
        round(results['poly']['accuracy'], 4),
        round(best_accuracy, 4)
    ],
    'CV Score': [
        round(results['linear']['cv_score'], 4),
        round(results['rbf']['cv_score'], 4),
        round(results['poly']['cv_score'], 4),
        round(grid_search.best_score_, 4)
    ]
})

summary['Test Accuracy (%)'] = (summary['Test Accuracy'] * 100).round(2)
print(summary.to_string(index=False))

print(f"\n✅ Best Configuration: {grid_search.best_params_}")
print(f"✅ Best Test Accuracy : {best_accuracy*100:.2f}%")

## 11. Save the Final Best Model

In [ ]:
# ── Save the Best SVM Model & Scaler using joblib ────────────────────────────
import joblib
import os

# Create a folder to store model files
os.makedirs('saved_model', exist_ok=True)

# Save the trained best SVM model
joblib.dump(best_svm, 'saved_model/best_svm_model.pkl')

# Save the scaler (IMPORTANT: must use same scaler when predicting new data)
joblib.dump(scaler, 'saved_model/scaler.pkl')

# Save the label encoder for 'type' column
joblib.dump(le, 'saved_model/label_encoder.pkl')

print("✅ Model saved  → saved_model/best_svm_model.pkl")
print("✅ Scaler saved → saved_model/scaler.pkl")
print("✅ Encoder saved→ saved_model/label_encoder.pkl")
print(f"\nModel details  : {best_svm}")
print(f"Best params    : {grid_search.best_params_}")

In [ ]:
# ── Verify Model Was Saved Correctly (Reload & Check) ────────────────────────
loaded_model  = joblib.load('saved_model/best_svm_model.pkl')
loaded_scaler = joblib.load('saved_model/scaler.pkl')

# Quick sanity check: predict on test set using loaded model
y_pred_loaded = loaded_model.predict(loaded_scaler.transform(X_test))
loaded_acc    = accuracy_score(y_test, y_pred_loaded)

print(f"✅ Model reloaded successfully!")
print(f"✅ Reloaded model accuracy: {loaded_acc * 100:.2f}%  (should match original)")

## 12. Sample Prediction — Test the Saved Model on New Data

In [ ]:
# ── Load saved model and scaler ───────────────────────────────────────────────
model  = joblib.load('saved_model/best_svm_model.pkl')
scaler = joblib.load('saved_model/scaler.pkl')

# ── Define column order (must match training) ─────────────────────────────────
feature_cols = [
    'type_encoded', 'fixed acidity', 'volatile acidity', 'citric acid',
    'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
    'density', 'pH', 'sulphates', 'alcohol'
]

# ── Helper function: predict a single wine sample ────────────────────────────
def predict_wine_quality(wine_type, fixed_acidity, volatile_acidity, citric_acid,
                         residual_sugar, chlorides, free_so2, total_so2,
                         density, pH, sulphates, alcohol):
    """
    Predicts whether a wine is 'Good' or 'Average/Poor'.
    wine_type: 'white' or 'red'
    Returns prediction label and confidence probabilities.
    """
    # Encode wine type: white=1, red=0 (matching LabelEncoder from training)
    type_encoded = 1 if wine_type.lower() == 'white' else 0

    # Create input as DataFrame with correct column names
    input_data = pd.DataFrame([[
        type_encoded, fixed_acidity, volatile_acidity, citric_acid,
        residual_sugar, chlorides, free_so2, total_so2,
        density, pH, sulphates, alcohol
    ]], columns=feature_cols)

    # Scale the input using the saved scaler
    input_scaled = scaler.transform(input_data)

    # Predict class and probability
    prediction  = model.predict(input_scaled)[0]
    probability = model.predict_proba(input_scaled)[0]

    label = '🍷 GOOD Wine (Quality ≥ 7)' if prediction == 1 else '🍶 Average / Poor Wine (Quality < 7)'
    return label, probability, prediction

print("✅ Prediction function ready!")

### SAMPLE 1 — A high-quality white wine (expected: GOOD)

In [ ]:
label, prob, pred = predict_wine_quality(
    wine_type       = 'white',
    fixed_acidity   = 6.8,
    volatile_acidity= 0.26,
    citric_acid     = 0.34,
    residual_sugar  = 2.1,
    chlorides       = 0.038,
    free_so2        = 32,
    total_so2       = 110,
    density         = 0.9920,
    pH              = 3.18,
    sulphates       = 0.62,
    alcohol         = 12.5
)

print("━━━ Sample 1: High-Quality White Wine ━━━")
print(f"Prediction   : {label}")
print(f"Confidence   : Poor/Average = {prob[0]*100:.1f}%  |  Good = {prob[1]*100:.1f}%")

### SAMPLE 2 — A low-quality red wine (expected: AVERAGE/POOR)

In [ ]:
label, prob, pred = predict_wine_quality(
    wine_type       = 'red',
    fixed_acidity   = 8.5,
    volatile_acidity= 0.72,
    citric_acid     = 0.10,
    residual_sugar  = 2.8,
    chlorides       = 0.095,
    free_so2        = 10,
    total_so2       = 55,
    density         = 0.9980,
    pH              = 3.42,
    sulphates       = 0.48,
    alcohol         = 9.5
)

print("━━━ Sample 2: Low-Quality Red Wine ━━━")
print(f"Prediction   : {label}")
print(f"Confidence   : Poor/Average = {prob[0]*100:.1f}%  |  Good = {prob[1]*100:.1f}%")

### SAMPLE 3 — Batch Prediction: 5 real samples from the test set

In [ ]:
print("━━━ Sample 3: Batch Prediction on 5 Test Samples ━━━\n")

# Take 5 random samples from the test set
sample_df    = X_test.sample(5, random_state=7).copy()
sample_true  = y_test.loc[sample_df.index].values

# Scale and predict
sample_scaled = scaler.transform(sample_df)
sample_pred   = model.predict(sample_scaled)
sample_proba  = model.predict_proba(sample_scaled)

# Display results
results_df = sample_df.reset_index(drop=True).copy()
results_df['Actual Label']     = ['Good' if v==1 else 'Average/Poor' for v in sample_true]
results_df['Predicted Label']  = ['Good' if v==1 else 'Average/Poor' for v in sample_pred]
results_df['Confidence (Good)']= [f"{p[1]*100:.1f}%" for p in sample_proba]
results_df['Match?']           = ['✅' if a==p else '❌'
                                   for a, p in zip(sample_true, sample_pred)]

# Show only key columns for readability
display_cols = ['alcohol', 'volatile acidity', 'sulphates',
                'Actual Label', 'Predicted Label', 'Confidence (Good)', 'Match?']
print(results_df[display_cols].to_string(index=False))

### SAMPLE 4 — Interactive: Enter your own wine values

In [ ]:
print("━━━ Sample 4: Custom Wine Input ━━━\n")
print("Edit the values below to test any wine:\n")

# Change these values to test your own wine
my_wine = {
    'wine_type'        : 'white',   # 'red' or 'white'
    'fixed_acidity'    : 7.0,
    'volatile_acidity' : 0.30,
    'citric_acid'      : 0.35,
    'residual_sugar'   : 5.0,
    'chlorides'        : 0.045,
    'free_so2'         : 40,
    'total_so2'        : 150,
    'density'          : 0.9940,
    'pH'               : 3.25,
    'sulphates'        : 0.55,
    'alcohol'          : 11.0
}

label, prob, pred = predict_wine_quality(**my_wine)

print("Input Wine Properties:")
for key, val in my_wine.items():
    print(f"  {key:<22}: {val}")
print(f"\n{'─'*45}")
print(f"Prediction   : {label}")
print(f"Confidence   : Poor/Average = {prob[0]*100:.1f}%  |  Good = {prob[1]*100:.1f}%")
print(f"{'─'*45}")